<center>

# **Get Artifacts from OneLake Catalog**

</center>  

### Purpose
This notebook demonstrates how to get artifacts from the OneLake Catalog programmatically.  

**_Disclaimer:_** This solution uses a non-documented and internal Microsoft endpoint for fetching OneLake Catalog artifacts across workspaces. Since this is not an officially supported API, it may change without notice, which could impact the functionality of this approach. Use it with that in mind, and feel free to experiment!
Also these APIs should be run in the context of a User Principal as support for Service Principal on internal endpoints are not guaranteed.


### Configuration
The following cell define which artifact types are to be included in the artifact lookup.


In [ ]:
filterByTypes = [
    "Model",
    "Sql",
    "Lakehouse",
    "SqlAnalyticsEndpoint",
    "SQLDbNative",
    "Warehouse",
    "Datawarehouse",
]

### Notebook functionality
Below cells define functions etc. for setting workspace icons.

In [ ]:
import requests, re, time

# Get token for the running identity via notebookutils
token = notebookutils.credentials.getToken("pbi")

### Get cluster URL for use in metadata endpoints (unsupported endpoints)
def get_cluster_url():
    for attempt in range(2):  # initial attempt + 1 retry
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
        }

        response = requests.get("https://api.powerbi.com/v1.0/myorg/capacities", headers=headers)
        response.raise_for_status()
        response = response.json()
        
        match = re.match(
            r"(https://[^/]+/)",
            response.get("@odata.context", "")
        )

        if match and "redirect.analysis.windows.net" in match.group(1):
            return match.group(1)

        response = requests.get("https://api.powerbi.com/v1.0/myorg/datasets", headers=headers)
        response.raise_for_status()
        response = response.json()

        match = re.match(r"(https://[^/]+/)", response.get("@odata.context", ""))

        if match and "redirect.analysis.windows.net" in match.group(1):
            return match.group(1)

        # Only wait before retrying
        if attempt == 0:
            time.sleep(1)

    return None

In [ ]:
### Set cluster url based on response from Power BI API call. Overwrite with manual value if required.
CLUSTER_BASE_URL = get_cluster_url()
print(f"Using base URL: {CLUSTER_BASE_URL}")

In [ ]:
from collections import defaultdict

# Get artifacts
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}

payload = {
    "supportedTypes": filterByTypes,
    "isParentChildSupported": True,
}
response = requests.post(f"{CLUSTER_BASE_URL}/metadata/datahub/V2/artifacts", headers=headers, json=payload)
response.raise_for_status()

# Group artifacts by workspace
workspaces = defaultdict(list)
for item in response.json():
    workspace_name = item.get("workspaceName", "Unknown")
    workspaces[workspace_name].append(item)

# Print summary header
total_artifacts = sum(len(items) for items in workspaces.values())
print(f"Found {total_artifacts} top-level artifacts across {len(workspaces)} workspaces")
print("=" * 90)

# Iterate through each workspace and its artifacts
for workspace_name in sorted(workspaces.keys()):
    items = workspaces[workspace_name]
    workspace_id = items[0].get("workspaceObjectId", "")
    print(f"\n📁 {workspace_name} ({workspace_id})")
    print("-" * 90)

    for item in items:
        artifact = item.get("artifact", {})
        name = item.get("displayName", "")
        artifact_type = artifact.get("artifactType") or ("Semantic Model" if item.get("artifactType") == 3 else "Unknown")
        object_id = item.get("artifactObjectId", "")
        region = item.get("region", "")

        print(f"  • {name}  [{artifact_type}] ({object_id})")
        
        # Children (e.g. SqlAnalyticsEndpoint under a Lakehouse)
        children = item.get("artifactChildren", []) or []
        if children:
            for child in children:
                child_artifact = child.get("artifact", {})
                child_artifact = child.get("artifact", {})
                child_name = child.get("displayName", "")
                child_type = child_artifact.get("artifactType") or ("Semantic Model" if child.get("artifactType") == 3 else "Unknown")
                child_id = child.get("artifactObjectId", "")
                print(f"    └─ {child_name}  [{child_type}]  {child_id}")

print("\n" + "=" * 90)

# Quick summary by artifact type
type_counts = defaultdict(int)
for items in workspaces.values():
    for item in items:
        atype = item.get("artifact", {}).get("artifactType") or ("Semantic Model" if item.get("artifactType") == 3 else "Unknown")
        type_counts[atype] += 1

print("\nSummary by artifact type:")
for atype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {atype:.<30} {count}")